## Using the model to make prediction

In [1]:
import pandas as pd 
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np

C:\Users\elomark\anaconda3\envs\elo-TF\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


#### We Load the Model.h5, Scaler, label_encoder_gender, and one_hot_encoder_geo pickle files

In [9]:
#### We Load the Model.h5, Scaler,label_encoder_gender, and one_hot_encoder_geo pickle files
model = load_model('model.h5')

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('one_hot_encoder_geo.pkl','rb') as file:
    one_hot_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [6]:
## Example input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 43,
    'Tenure': 3,
    'Balance': 60000,
    'NumberOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
    
}

In [12]:
## One Hot Encoder 
geo_encoded = one_hot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

C:\Users\elomark\anaconda3\envs\elo-TF\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [14]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumberOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,43,3,60000,2,1,1,50000


In [17]:
input_df['Gender'] = label_encoder_gender.fit_transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumberOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,0,43,3,60000,2,1,1,50000


In [18]:
## Now we combine one hot encoder columns with inpt data
input_df = pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumberOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,43,3,60000,2,1,1,50000,1.0,0.0,0.0


In [20]:
input_df.rename(columns={'NumberOfProducts': 'NumOfProducts'}, inplace=True)

In [21]:
### Scaling the Data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516, -1.09499335,  0.3900109 , -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [22]:
### Prediction
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step


array([[0.12976451]], dtype=float32)

In [23]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.12976451)

In [24]:
if prediction_proba > 0.5:
    print("The customer is likely to leave.")

else:
    print("The customer is not likely to leave")

The customer is not likely to leave
